## Notebook35

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install requests-cache --quiet

In [ ]:
import time

import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

import requests
import requests_cache

In [ ]:
session = requests_cache.CachedSession(
    cache_name="data/turing/requests_cache",
    backend="sqlite",
    allowable_methods=("GET", "HEAD", "POST"),
    allowable_codes=(200, 404),
    expire_after=None
)

base_url = "https://en.wikipedia.org/w/api.php"
headers = {
    "User-Agent": "DataScienceBook/1.0 (yourname@example.edu)"
}

### Questions

This notebook builds its table over the network instead of reading one from
`data/`, the same way the chapter does. The population is every direct member
of `Category:Turing Award laureates` on English Wikipedia: the recipients of
the ACM's annual computing prize, which one of their own Wikipedia articles
describes as "informally considered the Nobel Prize of computer science." The
population is small enough to collect politely in one sitting. Unlike the
chapter's novelist categories, it comes from a single official award list, so
there is essentially no ambiguity about who belongs. The notebook produces two
tables: `laureates`, a spine with a `page_id` and a `doc_id` for every
recipient, and `texts`, each laureate's article body. Notebook36 reuses both
to collect the links, revision histories, page views, and language links the
rest of the chapter covers. Print results unless a question says otherwise.

1. In the first block, send a single request for `"Donald Knuth"`'s page
info (`action="query"`, `prop="info"`) with `session.get()` and print the
status code. In the second block, parse `.json()` and pull `page_id`,
`title`, `length`, and `touched` out of it into a one-row `pl.DataFrame`.

2. Wikimedia asks clients to send a descriptive `User-Agent`, request
serially, and cache what comes back. Also clear `session.cookies` before every
request: Wikimedia sends a `Vary: Cookie` header, so a leftover cookie would
defeat the cache on every rerun. In the first block, write a
`polite_get(url, params)` helper that clears cookies, calls `session.get()`,
raises on a bad status with `.raise_for_status()`, and sleeps 0.2 seconds only
when the response did not come from the cache. In the second block, call it
twice in a row for `"Barbara Liskov"`'s page info and print `.from_cache` for
both calls.

**Answer**:


3. In the first block, write `category_members(category)`: loop over
`list="categorymembers"` with `cmtype="page"` (excludes subcategories and
files) and `cmlimit="max"`, following the `continue` block until it
disappears. In the second block, run it for `"Category:Turing Award
laureates"`, build a `laureates` table with `page_id` and `doc_id` columns,
drop any duplicate `page_id` with `.unique(subset="page_id", keep="first")`,
sort by `doc_id`, and print the shape. In the third block print the first 5
rows; in the fourth, print the last 5.

**Answer**:


4. Filter `laureates` to titles containing a parenthesis with
`.filter(c.doc_id.str.contains(r"\("))`. Print the result. Why would
Wikipedia need extra text in some of these article titles but not the
other 75?

**Answer**:


5. Someone typing this population from memory would not always type the
canonical title. In the first block, send one `prop="info"` request with
`redirects=1` for `"Vinton Cerf"`, `"Ronald Rivest"`, `"Edsger Dijkstra"`,
and a fake title, `"Not A Real Turing Laureate Page"`, and print the
`redirects` block. In the second block, print the `pages` entries flagged
`missing`.

**Answer**:


6. Two of question 4's disambiguated laureates, `John McCarthy (computer
scientist)` and `David Patterson (computer scientist)`, are also the
*plain* form of a name some other Wikipedia article uses. In the first
block, request `prop="info"` with `redirects=1` for the plain titles `"John
McCarthy"` and `"David Patterson"` (no parentheses), and print the `pageid`
each resolves to. In the second block, filter `laureates` to those same two
people so you can compare those IDs against the `page_id` values it has on
file. In the third block, request `prop="extracts"` (`exintro=1`) for the
two plain-title page IDs and print the start of each extract. What happened,
and why wouldn't question 5's check have caught it?

**Answer**:


7. In the first block, loop over every `page_id` in `laureates`, request
`prop="extracts"` with `explaintext=1`, collect each `page_id` with its
plain-text `text`, join the result back to `laureates` to attach `doc_id`
(keeping `page_id` and `doc_id` as the first two columns), and print the
shape. In the second block, filter to `c.text == ""` and print how many
rows come back.

**Answer**:


8. Print the first 300 characters of Donald Knuth's collected text and
check it against what you already know about him.

**Answer**:


9. In the first block, add an `n_chars` column to `texts` with
`.str.len_chars()`, then sort descending and print the 5 laureates with the
longest collected articles. In the second block, sort ascending and print
the 5 shortest. What seems to drive the difference?

**Answer**:


10. Filter `texts` to rows whose `text` contains `"Bell Labs"` with
`.str.contains()`. In the first block, print the matching `doc_id` values;
in the second, print the count.

**Answer**:


11. Request `prop="info"` for every `page_id` in `laureates`, batching up to
50 IDs per request with `"|".join()`, and collect each page's `length` (the
size of the wiki markup, in bytes) into a `wiki_length` column. Join it onto
`texts` and compute `pl.corr()` between `n_chars` (our own extracted plain
text) and `wiki_length` (the server's own count of the raw markup).

**Answer**:
